> 🔑 **BẢN LỜI GIẢI** — chỉ mở khi đã tự làm và vẫn bí.

# Bài 1 · Thực hành — Mạng nơ-ron học thế nào

Notebook này đi kèm trang lý thuyết **Bài 1**. Mỗi phần ứng với một mục lý thuyết:

| Phần | Mục lý thuyết | Bạn sẽ tự viết |
|---|---|---|
| A | 2 · Dữ liệu là tensor | shape, reshape, hàm chuẩn hóa ảnh giống hệt torchvision |
| B | 3 · Neuron, Linear, ReLU | lớp Linear và ReLU bằng numpy |
| C | 4 · Softmax và cross-entropy | softmax ổn định số, cross-entropy |
| D | 5 · Gradient descent | vòng lặp gradient descent, hồi quy tuyến tính từ đầu |
| E | 6 · Backpropagation | backprop thủ công cho mạng 2 lớp, so với autograd |
| F | 7 · Ghép lại | cùng mạng đó viết bằng PyTorch với 5 bước của training loop |

**Cách làm:** chạy từ trên xuống. Viết code vào chỗ `# TODO`, rồi chạy ô **✅ Kiểm tra** ngay sau. Thấy ✅ là xong phần đó. Notebook chạy được trên CPU (không cần GPU), mở bằng Google Colab hoặc Jupyter đều được.

Quy tắc: ở phần A–E chỉ dùng **numpy** để viết lời giải. PyTorch chỉ dùng trong ô kiểm tra để so đáp án.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

np.random.seed(0); torch.manual_seed(0)
def close(a, b, tol=1e-6):
    return np.allclose(np.asarray(a, dtype=np.float64), np.asarray(b, dtype=np.float64), atol=tol, rtol=tol)
print("numpy", np.__version__, "| torch", torch.__version__)

## Phần A — Dữ liệu là tensor

Nhắc lại: PyTorch lưu một batch ảnh theo thứ tự **[N, C, H, W]** = [số ảnh, số kênh màu, cao, rộng]. Ảnh đọc từ file (PIL, OpenCV) thường ở dạng **[H, W, C]** với giá trị nguyên 0–255.

### A1. Shape và reshape

In [ ]:
x = np.random.rand(4, 3, 32, 32)   # một batch 4 ảnh màu 32×32

N, C, H = x.shape[0], x.shape[1], x.shape[2]

x_flat = x.reshape(x.shape[0], -1)

In [ ]:
# ✅ Kiểm tra A1
assert (N, C, H) == (4, 3, 32), f"Sai: {(N, C, H)}"
assert x_flat is not None and x_flat.shape == (4, 3072), "x_flat phải có shape (4, 3072)"
assert np.array_equal(x_flat[1], x[1].ravel()), "Ảnh thứ 2 bị trộn với ảnh khác — reshape phải giữ chiều batch đầu tiên"
print("✅ A1 đúng! 3 × 32 × 32 =", 3 * 32 * 32)

### A2. Tự viết `ToTensor` + `Normalize`

Viết hàm nhận một ảnh `uint8` shape **(H, W, 3)** và trả về mảng `float32` shape **(3, H, W)**, làm đúng 3 việc:
1. chia 255 để giá trị về 0–1
2. chuẩn hóa từng kênh: `(x - mean[c]) / std[c]`
3. đổi thứ tự trục từ (H, W, C) sang (C, H, W)

Kết quả phải trùng với `torchvision.transforms` mà Lab 1 dùng.

In [ ]:
MEAN = np.array([0.4914, 0.4822, 0.4465])
STD  = np.array([0.2470, 0.2435, 0.2616])

def to_tensor_normalize(img_uint8):
    x = img_uint8.astype(np.float32) / 255.0      # (H, W, 3), 0–1
    x = (x - MEAN) / STD                          # broadcast theo trục cuối = kênh màu
    return x.transpose(2, 0, 1).astype(np.float32)  # (3, H, W)

In [ ]:
# ✅ Kiểm tra A2
import torchvision.transforms as T
from PIL import Image
img = (np.random.rand(32, 32, 3) * 255).astype(np.uint8)
ours = to_tensor_normalize(img)
ref = T.Compose([T.ToTensor(), T.Normalize(tuple(MEAN), tuple(STD))])(Image.fromarray(img)).numpy()
assert ours.shape == (3, 32, 32), f"Shape phải là (3, 32, 32), đang là {ours.shape}"
assert ours.dtype == np.float32, "Kiểu dữ liệu phải là float32"
assert close(ours, ref, 1e-5), f"Lệch so với torchvision tối đa {np.abs(ours - ref).max():.4f}"
print("✅ A2 đúng! Pixel 200 ở kênh R sau chuẩn hóa =", round((200 / 255 - 0.4914) / 0.2470, 3))

## Phần B — Lớp Linear và ReLU

Lớp `nn.Linear(in_f, out_f)` lưu `weight` shape **(out_f, in_f)** và `bias` shape **(out_f,)**. Với một batch `x` shape (N, in_f):

$$ y = x W^\top + b \quad\Rightarrow\quad y \text{ có shape } (N, \text{out\_f}) $$

ReLU: $\mathrm{relu}(z) = \max(0, z)$, áp dụng cho từng phần tử.

In [ ]:
def linear(x, W, b):
    return x @ W.T + b

def relu(z):
    return np.maximum(0, z)

In [ ]:
# ✅ Kiểm tra B1, B2
# Ví dụ tính tay trong trang lý thuyết: x=[1,2], W=[[0.5,-1],[2,0.5]], b=[0,1]
z = linear(np.array([[1., 2.]]), np.array([[0.5, -1.], [2., 0.5]]), np.array([0., 1.]))
assert close(z, [[-1.5, 4.0]]), f"Ví dụ tính tay phải ra [[-1.5, 4.0]], đang là {z}"
assert close(relu(z), [[0., 4.]]), "relu([-1.5, 4]) phải là [0, 4]"

layer = nn.Linear(5, 3).double()
xb = np.random.randn(8, 5)
ours = linear(xb, layer.weight.detach().numpy(), layer.bias.detach().numpy())
assert close(ours, layer(torch.from_numpy(xb)).detach().numpy()), "Không khớp với nn.Linear"
print("✅ B1, B2 đúng! Output nn.Linear(5,3) trên batch 8:", ours.shape)

### B3. Vì sao cần hàm kích hoạt?

Hai lớp Linear chồng lên nhau **không có ReLU ở giữa** vẫn chỉ là một lớp Linear. Hãy chứng minh bằng code: tìm `W` và `b` sao cho `linear(linear(x, W1, b1), W2, b2) == linear(x, W, b)` với mọi `x`.

Gợi ý: khai triển $(xW_1^\top + b_1)W_2^\top + b_2$.

In [ ]:
W1, b1 = np.random.randn(4, 3), np.random.randn(4)
W2, b2 = np.random.randn(2, 4), np.random.randn(2)

W = W2 @ W1          # (2, 3)
b = W2 @ b1 + b2     # (2,)

In [ ]:
# ✅ Kiểm tra B3
xs = np.random.randn(10, 3)
assert W is not None and W.shape == (2, 3), "W phải có shape (2, 3)"
assert close(linear(linear(xs, W1, b1), W2, b2), linear(xs, W, b)), "Chưa đúng"
two_with_relu = linear(relu(linear(xs, W1, b1)), W2, b2)
print("✅ B3 đúng! Hai lớp Linear = một lớp Linear.")
print("   Có ReLU ở giữa thì còn bằng không?", close(two_with_relu, linear(xs, W, b)), "→ ReLU làm mạng 'sâu' thật sự")

## Phần C — Softmax và cross-entropy

$$ \mathrm{softmax}(z)_i = \frac{e^{z_i}}{\sum_j e^{z_j}} \qquad\qquad \mathcal{L} = -\log p_{y} $$

**Mẹo ổn định số:** $e^{1000}$ tràn số thành `inf`. Vì $\mathrm{softmax}(z) = \mathrm{softmax}(z - c)$ với mọi hằng số $c$, ta trừ đi $\max(z)$ trước khi lấy mũ. Mọi thư viện đều làm vậy.

### C1. Softmax ổn định

In [ ]:
def softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

In [ ]:
# ✅ Kiểm tra C1
p = softmax(np.array([[2., 1., 0.]]))
assert close(p, [[0.66524096, 0.24472847, 0.09003057]]), f"softmax([2,1,0]) sai: {p}"
big = softmax(np.array([[1000., 1001.], [-1000., -1000.]]))
assert np.all(np.isfinite(big)), "Bị nan/inf với logits lớn — bạn đã trừ max chưa?"
assert close(big, [[0.26894142, 0.73105858], [0.5, 0.5]])
zr = np.random.randn(6, 10)
assert close(softmax(zr), F.softmax(torch.from_numpy(zr), dim=1).numpy())
print("✅ C1 đúng! softmax([2,1,0]) =", p.round(3))

### C2. Cross-entropy cho cả batch

`y` là mảng nhãn số nguyên shape (N,). Loss của batch là **trung bình** loss từng ảnh. Lấy xác suất của đúng lớp từng hàng bằng fancy indexing: `p[np.arange(N), y]`.

In [ ]:
def cross_entropy(logits, y):
    p = softmax(logits)
    N = logits.shape[0]
    return -np.log(p[np.arange(N), y]).mean()

In [ ]:
# ✅ Kiểm tra C2
assert close(cross_entropy(np.array([[2., 1., 0.]]), np.array([0])), 0.40760596), "Ví dụ tính tay phải ra ≈ 0.408"
lg, yy = np.random.randn(16, 10), np.random.randint(0, 10, 16)
assert close(cross_entropy(lg, yy), F.cross_entropy(torch.from_numpy(lg), torch.from_numpy(yy)).item())
uniform = cross_entropy(np.zeros((4, 10)), np.array([0, 3, 5, 9]))
assert close(uniform, np.log(10)), "Logits bằng nhau với 10 lớp phải cho loss = ln 10"
print(f"✅ C2 đúng! Model 'đoán bừa' 10 lớp có loss = ln 10 = {uniform:.4f}")
print("   (Đây là con số ≈ 2.30 bạn sẽ thấy ở epoch đầu của Lab 1)")

## Phần D — Gradient descent

Quy tắc cập nhật: $w \leftarrow w - \eta \cdot f'(w)$, với $\eta$ là learning rate.

### D1. Vòng lặp gradient descent
Viết hàm chạy `steps` bước, trả về **list các giá trị w** (gồm cả w0 ở đầu, nên dài `steps + 1`).

In [ ]:
def gradient_descent(df, w0, lr, steps):
    ws = [w0]
    w = w0
    for _ in range(steps):
        w = w - lr * df(w)
        ws.append(w)
    return ws

In [ ]:
# ✅ Kiểm tra D1 — f(w) = (w - 3)², f'(w) = 2(w - 3)
df = lambda w: 2 * (w - 3)
ws = gradient_descent(df, w0=0.0, lr=0.1, steps=100)
assert len(ws) == 101, "Phải trả về steps + 1 giá trị"
assert close(ws[1], 0.6) and close(ws[2], 1.08), f"2 bước đầu phải là 0.6 và 1.08 (giống bài tính tay), đang là {ws[1]}, {ws[2]}"
assert abs(ws[-1] - 3) < 1e-6, "Sau 100 bước phải hội tụ về 3"

fig, ax = plt.subplots(figsize=(8, 3))
for lr in [0.01, 0.1, 0.9, 1.05]:
    ax.plot(gradient_descent(df, 0.0, lr, 30), label=f"lr={lr}")
ax.axhline(3, c="gray", ls="--"); ax.set_ylim(-3, 9); ax.set_xlabel("bước"); ax.set_ylabel("w"); ax.legend(); plt.show()
print("✅ D1 đúng! Nhìn biểu đồ: lr nhỏ thì chậm, lr=0.9 thì dao động, lr=1.05 thì văng ra (phân kỳ).")

**Câu hỏi suy nghĩ:** với $f(w) = (w-3)^2$, vì sao mọi `lr > 1` đều phân kỳ? (Gợi ý: viết $w_{t+1} - 3$ theo $w_t - 3$.)

### D2. Hồi quy tuyến tính từ con số 0

Dữ liệu thật tuân theo $y = 2x + 1$ cộng nhiễu. Model: $\hat{y} = wx + b$. Loss MSE: $\mathcal{L} = \frac{1}{N}\sum (\hat{y} - y)^2$.

Đạo hàm (tự kiểm chứng bằng quy tắc dây chuyền):

$$ \frac{\partial \mathcal{L}}{\partial w} = \frac{2}{N}\sum(\hat{y}-y)\,x \qquad \frac{\partial \mathcal{L}}{\partial b} = \frac{2}{N}\sum(\hat{y}-y) $$

In [ ]:
rng = np.random.default_rng(1)
X = rng.uniform(-2, 2, 100)
Y = 2 * X + 1 + rng.normal(0, 0.3, 100)

w, b, lr = 0.0, 0.0, 0.1
losses = []
for step in range(200):
    y_hat = w * X + b
    err = y_hat - Y
    losses.append((err ** 2).mean())
    dw = 2 * (err * X).mean()
    db = 2 * err.mean()
    w -= lr * dw
    b -= lr * db

In [ ]:
# ✅ Kiểm tra D2
assert len(losses) == 200, "Mỗi bước phải ghi một giá trị loss"
assert abs(w - 2) < 0.1 and abs(b - 1) < 0.1, f"Phải tìm được w≈2, b≈1, đang là w={w:.3f}, b={b:.3f}"
assert losses[-1] < 0.15, "Loss cuối phải gần mức nhiễu 0.3² = 0.09"
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3))
a1.plot(losses); a1.set_yscale("log"); a1.set_title("Loss theo bước")
a2.scatter(X, Y, s=8); xs = np.linspace(-2, 2, 2); a2.plot(xs, w * xs + b, c="C1"); a2.set_title(f"y = {w:.2f}x + {b:.2f}")
plt.show()
print(f"✅ D2 đúng! Học được w = {w:.3f}, b = {b:.3f}")

## Phần E — Backpropagation

### E1. Ví dụ tính tay, kiểm bằng autograd
Cùng đồ thị tính toán như trang lý thuyết, nhưng với số mới: $x=1.5,\ w=2,\ b=0.5,\ y=2$, learning rate $0.1$

$$ z = wx + b \;\rightarrow\; a = \mathrm{relu}(z) \;\rightarrow\; \mathcal{L} = (a - y)^2 $$

Điền các giá trị bạn tính tay (không dùng code để tính), rồi chạy ô kiểm tra. Ô kiểm tra sẽ dùng **autograd** của PyTorch để chấm.

In [ ]:
my_L      = 2.25
my_dL_dw  = 4.5
my_dL_db  = 3.0
my_w_new  = 1.55

In [ ]:
# ✅ Kiểm tra E1 — autograd chấm bài
w_t = torch.tensor(2.0, requires_grad=True)
b_t = torch.tensor(0.5, requires_grad=True)
L_t = (torch.relu(w_t * 1.5 + b_t) - 2.0) ** 2
L_t.backward()
for name, mine, ref in [("L", my_L, L_t.item()), ("dL/dw", my_dL_dw, w_t.grad.item()),
                        ("dL/db", my_dL_db, b_t.grad.item()), ("w mới", my_w_new, 2.0 - 0.1 * w_t.grad.item())]:
    assert mine is not None, f"Chưa điền {name}"
    assert close(mine, ref), f"{name}: bạn ghi {mine}, đúng là {ref}"
print("✅ E1 đúng! Bạn vừa làm đúng việc mà loss.backward() làm.")

### E2. Gradient bị cộng dồn — vì sao cần `zero_grad()`
Chạy ô dưới (đã viết sẵn) và đọc kết quả.

In [ ]:
w_t = torch.tensor(3.0, requires_grad=True)
for i in range(3):
    L_t = (torch.relu(w_t * 2.0 + 1.0) - 5.0) ** 2
    L_t.backward()
    print(f"lần backward thứ {i+1}: w.grad = {w_t.grad.item()}")
print("→ Gradient CỘNG DỒN: 8, 16, 24. Nếu không gọi optimizer.zero_grad(), mỗi bước sẽ dùng gradient sai.")

### E3. Backprop thủ công cho mạng 2 lớp (phần khó nhất)

Bài toán: phân loại điểm 2D thành 2 lớp nằm trên 2 vòng tròn lồng nhau. Một đường thẳng không tách được, nên cần mạng có lớp ẩn.

Mạng: `X (N,2) → Linear(2,16) → ReLU → Linear(16,2) → softmax → cross-entropy`

**Forward:**
```
z1 = X @ W1.T + b1        # (N,16)
h  = relu(z1)             # (N,16)
z2 = h @ W2.T + b2        # (N,2)  logits
L  = cross_entropy(z2, y)
```
**Backward** (đi ngược, mỗi dòng là một lần dùng quy tắc dây chuyền):
```
dz2 = (softmax(z2) - onehot(y)) / N     # đạo hàm của CE theo logits — công thức đẹp nhất trong ML
dW2 = dz2.T @ h          db2 = dz2.sum(0)
dh  = dz2 @ W2
dz1 = dh * (z1 > 0)                     # ReLU cho gradient đi qua khi z1 > 0, chặn khi z1 ≤ 0
dW1 = dz1.T @ X          db1 = dz1.sum(0)
```
Hãy viết hàm `forward_backward` trả về loss và dict gradient. Ô kiểm tra so từng gradient với autograd.

In [ ]:
def make_circles(n=400, seed=0):
    r = np.random.default_rng(seed)
    y = r.integers(0, 2, n)
    radius = np.where(y == 0, 0.5, 1.0) + r.normal(0, 0.08, n)
    ang = r.uniform(0, 2 * np.pi, n)
    return np.stack([radius * np.cos(ang), radius * np.sin(ang)], 1), y

Xc, yc = make_circles()
plt.figure(figsize=(4, 4)); plt.scatter(Xc[:, 0], Xc[:, 1], c=yc, cmap="coolwarm", s=10); plt.axis("equal"); plt.show()

def init_params(seed=0):
    r = np.random.default_rng(seed)
    return {"W1": r.normal(0, 1.0, (16, 2)), "b1": np.zeros(16),
            "W2": r.normal(0, 0.5, (2, 16)), "b2": np.zeros(2)}

In [ ]:
def forward_backward(params, X, y):
    W1, b1, W2, b2 = params["W1"], params["b1"], params["W2"], params["b2"]
    N = X.shape[0]
    z1 = X @ W1.T + b1
    h = relu(z1)
    z2 = h @ W2.T + b2
    loss = cross_entropy(z2, y)

    onehot = np.zeros_like(z2); onehot[np.arange(N), y] = 1
    dz2 = (softmax(z2) - onehot) / N
    dW2 = dz2.T @ h
    db2 = dz2.sum(0)
    dh = dz2 @ W2
    dz1 = dh * (z1 > 0)
    dW1 = dz1.T @ X
    db1 = dz1.sum(0)
    grads = {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2}
    return loss, grads

In [ ]:
# ✅ Kiểm tra E3 — so với autograd
params = init_params()
loss, grads = forward_backward(params, Xc, yc)
assert loss is not None, "Chưa tính loss"
tp = {k: torch.tensor(v, requires_grad=True) for k, v in params.items()}
z2_t = torch.relu(torch.from_numpy(Xc) @ tp["W1"].T + tp["b1"]) @ tp["W2"].T + tp["b2"]
loss_t = F.cross_entropy(z2_t, torch.from_numpy(yc)); loss_t.backward()
assert close(loss, loss_t.item()), f"Loss lệch: {loss} vs {loss_t.item()}"
for k in params:
    assert grads[k] is not None, f"Chưa tính gradient {k}"
    assert grads[k].shape == params[k].shape, f"grad {k} phải có shape {params[k].shape}, đang là {grads[k].shape}"
    assert close(grads[k], tp[k].grad.numpy()), f"Gradient {k} lệch autograd tối đa {np.abs(grads[k] - tp[k].grad.numpy()).max():.2e}"
print("✅ E3 đúng! Cả 4 gradient khớp autograd đến 1e-6. Bạn đã tự viết được backprop.")

### E4. Train mạng bằng gradient bạn vừa viết

In [ ]:
params = init_params()
lr = 0.5
hist = []
for step in range(2000):
    loss, grads = forward_backward(params, Xc, yc)
    hist.append(loss)
    for k in params:
        params[k] -= lr * grads[k]

In [ ]:
# ✅ Kiểm tra E4
def predict(params, X):
    h = relu(X @ params["W1"].T + params["b1"])
    return (h @ params["W2"].T + params["b2"]).argmax(1)
acc = (predict(params, Xc) == yc).mean()
assert len(hist) == 2000 and hist[-1] < hist[0] * 0.3, "Loss phải giảm mạnh"
assert acc > 0.95, f"Accuracy {acc:.1%} — phải > 95%"

gx, gy = np.meshgrid(np.linspace(-1.5, 1.5, 200), np.linspace(-1.5, 1.5, 200))
zz = predict(params, np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4))
a1.plot(hist); a1.set_title("Loss"); a1.set_xlabel("bước")
a2.contourf(gx, gy, zz, alpha=0.3, cmap="coolwarm"); a2.scatter(Xc[:, 0], Xc[:, 1], c=yc, cmap="coolwarm", s=8)
a2.set_title(f"Ranh giới học được — acc {acc:.1%}"); a2.set_aspect("equal"); plt.show()
print(f"✅ E4 đúng! Mạng tự viết bằng numpy đạt {acc:.1%}")

## Phần F — Ghép lại bằng PyTorch

Cùng mạng ở E3, nhưng giờ để PyTorch lo backward. Đây chính là khung training loop bạn sẽ dùng ở Lab 1 và mọi dự án sau.

Năm bước (học thuộc):
```
optimizer.zero_grad()
logits = model(x)
loss = criterion(logits, y)
loss.backward()
optimizer.step()
```

In [ ]:
class TwoLayerNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 2))

    def forward(self, x):
        return self.net(x)

torch.manual_seed(0)
X_t = torch.tensor(Xc, dtype=torch.float32)
y_t = torch.tensor(yc)
model = TwoLayerNet()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

for step in range(2000):
    optimizer.zero_grad()
    logits = model(X_t)
    loss = criterion(logits, y_t)
    loss.backward()
    optimizer.step()

In [ ]:
# ✅ Kiểm tra F
with torch.no_grad():
    acc_t = (model(X_t).argmax(1) == y_t).float().mean().item()
n_params = sum(p.numel() for p in model.parameters())
assert n_params == 16 * 2 + 16 + 2 * 16 + 2, f"Số tham số phải là 82, đang là {n_params}"
assert acc_t > 0.95, f"Accuracy {acc_t:.1%} — phải > 95%. Kiểm tra lại thứ tự 5 bước."
print(f"✅ F đúng! PyTorch đạt {acc_t:.1%} với {n_params} tham số — giống hệt bản numpy, nhưng chỉ 5 dòng.")
print("🎉 Hoàn thành thực hành Bài 1. Gửi Claude: câu trả lời câu hỏi suy nghĩ ở D1 + 3 câu dưới đây.")

## Câu hỏi nộp bài
1. Ở D1, vì sao `lr > 1` làm $f(w) = (w-3)^2$ phân kỳ?
2. Ở E3, nếu một neuron ẩn có `z1 ≤ 0` với **mọi** điểm dữ liệu thì gradient của nó bằng bao nhiêu? Hiện tượng này gọi là gì?
3. Đổi `lr` ở E4 thành `5.0` rồi `0.01`. Mô tả đường loss trong từng trường hợp và giải thích.

Tiếp theo: **Bài 2 · Train mạng thật** (mini-batch, optimizer, CNN, BatchNorm, Dropout, overfitting), sau đó làm Lab 1.